<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/08_skills.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08 · Skills: progressive disclosure of procedures

`AGENTS.md` is always loaded. That is fine for a page of house style and ruinous for a dozen
detailed procedures — you would pay for every one of them on every model call, and the model
would wade through eleven irrelevant workflows to find the one that applies.

A **skill** is a procedure the agent loads *only when it is relevant*. Its name and description
are always visible; the body arrives on demand.

**New in this lesson:** `SKILL.md` anatomy, `skills=[...]`, bundled files, source layering

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "git+https://github.com/langchain-samples/langsmith-studio-nb.git"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-08-skills"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. Anatomy of a skill

A skill is a **directory** containing `SKILL.md`: YAML frontmatter (a `name` and a
`description`) followed by the procedure itself.

Because it is a directory, you have to create it before writing into it — `%%writefile` will not
create parent directories for you.

In [ ]:
!mkdir -p agent_home/skills/weekly-report
!mkdir -p agent_home/skills/incident-postmortem

In [ ]:
%%writefile agent_home/skills/weekly-report/SKILL.md
---
name: weekly-report
description: Compile the weekly support summary. Use when asked for a weekly report, a week in review, or a summary of the last seven days of tickets.
---

# Weekly support report

## When to use
The user asks for a weekly report, a week-in-review, or a summary of recent ticket activity.

## Procedure
1. Count tickets by category: damage, delivery, billing, other.
2. Identify the single most common root cause.
3. Note any ticket that has been reopened more than once — these are the real problems.
4. Write the report using exactly this structure:

```
## Week in review
**Volume:** N tickets (up/down X% on last week)
**Top issue:** <one sentence>
**Needs attention:** <bulleted list, or "nothing outstanding">
**Recommendation:** <one concrete action for next week>
```

## Rules
- Never speculate about causes you cannot support from ticket text.
- If volume is under 5 tickets, say the sample is too small rather than computing a percentage.

In [ ]:
%%writefile agent_home/skills/incident-postmortem/SKILL.md
---
name: incident-postmortem
description: Write a blameless postmortem after an outage or service incident. Use when asked about an incident, an outage, downtime, or what went wrong.
---

# Incident postmortem

## When to use
An outage or service incident has occurred and the user wants it written up.

## Procedure
1. Establish the timeline first: detection, escalation, mitigation, resolution. Times, not adjectives.
2. Separate **trigger** (what started it) from **cause** (why the system was vulnerable).
3. Write the report:

```
## Incident: <short title>
**Duration:** <start> - <end> (<total>)
**Impact:** <who was affected and how>
**Trigger:** <the immediate event>
**Root cause:** <the underlying weakness>
**Timeline:** <bulleted, timestamped>
**Action items:** <each with an owner and a due date>
```

## Rules
- Blameless: name systems and processes, never individuals.
- Every action item needs an owner. "The team" is not an owner.
- If the root cause is unknown, say so explicitly rather than guessing.

In [ ]:
!find agent_home/skills -type f

---

## 2. Hand them to an agent

`skills=[...]` takes **source paths** relative to the backend root, not individual skills. Every
skill directory under a source is discovered automatically.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

skilled_agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a support team assistant.",
    backend=FilesystemBackend(root_dir="agent_home"),
    skills=["/skills/"],
)
skilled_agent

In [ ]:
from langsmith_studio_nb import start_studio

start_studio("skilled_agent")

Example prompts:
> We had 14 tickets this week: 6 damage, 4 delivery, 3 billing, 1 other. T-1 has been reopened twice. Give me the weekly report.

> Our order lookup service was down from 09:15 to 10:40 today. A bad config push removed the database credentials, and we found it when support tickets spiked. Write this up.

Two different questions, two different procedures, **one agent** — and at no point were both full
procedures in context at the same time.

### That is progressive disclosure

Look at the first run in Studio. Before the skill loaded, the system prompt carried only each
skill's `name` and `description` — a couple of dozen tokens. Then a tool result appears containing
the full body of `weekly-report/SKILL.md`, and nothing at all from `incident-postmortem`.

The mechanism is deliberately unglamorous: skills are files, and loading one is a file read.
There is no special retrieval, no embedding, no vector search. The model reads a menu, then reads
one page.

Which makes the `description` the single most important line in the file: **it is the entire
basis on which the model decides whether to look further.** Write it for a chooser, not a reader
— say *when to use this*, not *what this is*.

---

## 3. Skills can ship files

A skill directory can hold more than `SKILL.md`. Templates, checklists, example documents,
reference data — anything the procedure tells the agent to go and read.

In [ ]:
!mkdir -p agent_home/skills/customer-apology

In [ ]:
%%writefile agent_home/skills/customer-apology/SKILL.md
---
name: customer-apology
description: Draft an apology to a customer after a service failure. Use when a customer is upset, an order was mishandled, or compensation is being offered.
---

# Customer apology

## When to use
A customer has been let down and needs a written apology.

## Procedure
1. Read `template.md` in this skill directory. Follow its structure exactly.
2. Fill every placeholder. Never leave a `<...>` in a sent message.
3. Name the specific failure. Generic regret reads as evasion.

## Rules
- One apology, at the start. Repeating it sounds insincere.
- Always state what will happen next, with a timeframe.

In [ ]:
%%writefile agent_home/skills/customer-apology/template.md
Dear <first name>,

I'm sorry that <specific thing that went wrong>.

<One sentence explaining what happened, without jargon or blame.>

Here's what happens next:
- <action> by <date>
- <action> by <date>

<If compensation applies: state exactly what it is and when it arrives.>

Thank you for your patience.
<agent name>
Customer Support

In [ ]:
apology_agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a support team assistant. Sign messages as 'Sam'.",
    backend=FilesystemBackend(root_dir="agent_home"),
    skills=["/skills/"],
)
apology_agent

In [ ]:
start_studio("apology_agent")

Example prompt:
> Avery's standing desk (order 1042) arrived cracked, and our first reply took six days. We're refunding in full. Write the apology.

Two file reads deep: the agent read `SKILL.md`, which told it to read `template.md`, and followed
the structure. The template never entered the system prompt — it was fetched mid-task.

---

## 4. Layering sources

`skills=[...]` takes a list, loaded in order, and **later sources win on name collisions**. That
gives you an override chain without any special machinery:

```python
skills=["/skills/base/", "/skills/team/", "/skills/user/"]
```

A company-wide `weekly-report` in `base`, a regional variation in `team`, and one person's tweak
in `user` — the agent sees exactly one `weekly-report`, the most specific one.

---

## 5. Skill, tool, subagent, or AGENTS.md?

The four mechanisms blur together until you ask **what kind of thing** you are adding:

| You are adding | Use | Because |
|---|---|---|
| A **procedure** — steps a competent person could follow | **Skill** | Instructions, needed occasionally, too long to always carry |
| A **capability** — something the model fundamentally cannot do | **Tool** | Reading a database is not a matter of instructions |
| **Isolation** — bulky work that would flood the context | **Subagent** | You want the result, not the intermediate mess |
| **Policy** — how everything should always be done | **`AGENTS.md`** | It applies to every request, so paying every time is correct |

The recurring test is **frequency versus size**. Small and always relevant → `AGENTS.md`. Large
and occasionally relevant → skill. Not a matter of instructions at all → tool. Too bulky to keep
→ subagent.

The common mistake is writing a tool when you needed a skill: wrapping a *procedure* in Python so
the agent cannot adapt it, when a `SKILL.md` would have let it.

---

## 📌 Key takeaways

- A skill is a **procedure** the agent loads only when it is relevant.
- The `description` is the entire selection mechanism — write it for a chooser: *when to use this*.
- Progressive disclosure means names and descriptions are always in context; bodies are read on demand.
- A skill is a **directory**, so `mkdir -p` before `%%writefile` — it will not create parents.
- Skills can bundle templates and reference files the procedure tells the agent to read.
- Later skill sources override earlier ones by name, giving you base → team → user layering.
- Procedure → skill, capability → tool, isolation → subagent, always-on policy → `AGENTS.md`.

---

## ➡️ Next

**[09 · Under the hood](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/09_under_the_hood.ipynb)**

That is the whole harness: files, tools, subagents, middleware, memory, skills. Time to open the
box and see that it was `create_agent` plus a middleware stack all along.